# Backpropagation: hands-on (optional)
**LLMs From Basics · Lesson P2.3**

You'll build backprop yourself, in three stages:
1. by hand, for the tiny network from the lesson,
2. checked against **numerical** gradients (the "nudge each parameter" method),
3. as a tiny **automatic** engine: numbers that remember how they were made and can run the chain rule backwards. It's the same idea as PyTorch's autograd, in about 40 lines.

Each exercise has a cell with a `# TODO`, followed by a **check** cell with `assert`s. Run it; silence (or a ✓) means success. Solutions are at the very bottom. Try first: *struggling to recall is what makes it stick.*

Only `numpy` is needed (pre-installed on Colab).

In [ ]:
import math
import numpy as np
np.random.seed(0)
print("ready")

## 1 · Backprop by hand
The network from the lesson:

$$z = w\,x, \qquad h = \mathrm{relu}(z), \qquad \hat y = v\,h, \qquad L = (\hat y - y)^2$$

Write the forward pass, then the backward pass, one node at a time: **upstream gradient × local derivative**.
Return the loss and the gradients `dw` $=\frac{\partial L}{\partial w}$ and `dv` $=\frac{\partial L}{\partial v}$.

In [ ]:
def forward_backward(x, w, v, y):
    # forward
    z = w * x
    h = max(0.0, z)
    yhat = v * h
    L = (yhat - y) ** 2
    # TODO: backward. Compute dyhat, then dv and dh, then dz, then dw.
    # Hint: relu's local derivative is 1 if z > 0 else 0.
    raise NotImplementedError("Exercise 1")
    return L, dw, dv

**Check 1.** The numbers from the lesson's animation and faded example.

In [ ]:
L, dw, dv = forward_backward(x=1.5, w=2.0, v=0.5, y=2.5)
assert abs(L - 1.0) < 1e-12 and abs(dw - (-1.5)) < 1e-12 and abs(dv - (-6.0)) < 1e-12
L, dw, dv = forward_backward(x=2.0, w=1.0, v=-1.0, y=1.0)
assert (L, dw, dv) == (9.0, 12.0, -12.0)
L, dw, dv = forward_backward(x=1.0, w=-2.0, v=3.0, y=1.0)   # z < 0: relu blocks the gradient
assert dw == 0.0, "when z < 0, no gradient reaches w"
print("✓ exercise 1")

## 2 · Check with numerical gradients
The slow-but-simple way: nudge one input by a tiny $\varepsilon$ in each direction and measure the slope (a **central difference**):

$$\frac{\partial L}{\partial w} \approx \frac{L(w + \varepsilon) - L(w - \varepsilon)}{2\varepsilon}$$

Every serious autograd implementation is tested like this. Implement it for any function `f` of a list of numbers.

In [ ]:
def numerical_grad(f, params, eps=1e-6):
    # f takes a list of numbers and returns a scalar.
    # TODO: return a list with the central-difference slope for each entry of params
    raise NotImplementedError("Exercise 2")

**Check 2.** Compare your hand-written backprop with the numerical slopes at 20 random points.

In [ ]:
for _ in range(20):
    x, w, v, y = np.random.uniform(-2, 2, size=4)
    if abs(w * x) < 1e-3:      # relu's kink: the slope isn't defined exactly at 0
        continue
    _, dw, dv = forward_backward(x, w, v, y)
    nw, nv = numerical_grad(lambda p: forward_backward(x, p[0], p[1], y)[0], [w, v])
    assert abs(dw - nw) < 1e-5 and abs(dv - nv) < 1e-5, (dw, nw, dv, nv)
print("✓ exercise 2: backprop matches the numerical gradients")

## 3 · A tiny autograd engine
Writing the backward pass by hand doesn't scale. Instead, let every number **remember how it was made**.
Each `Value` stores its data, its gradient `.grad`, the values it came from (`_prev`) and a `_backward` function that pushes its own gradient to its parents using the **local derivative**.

`__add__` is done for you. Notice the `+=`: a value used in several places receives gradient from each of them, and they **add up**.

Implement `__mul__`, `relu` and `__pow__` (for a constant exponent) in the same style.

In [ ]:
class Value:
    def __init__(self, data, _prev=(), _op=""):
        self.data = float(data)
        self.grad = 0.0
        self._prev = _prev
        self._op = _op
        self._backward = lambda: None

    def __add__(self, other):
        other = other if isinstance(other, Value) else Value(other)
        out = Value(self.data + other.data, (self, other), "+")
        def _backward():
            self.grad += 1.0 * out.grad      # local derivative of a + b w.r.t. a is 1
            other.grad += 1.0 * out.grad
        out._backward = _backward
        return out

    def __mul__(self, other):
        other = other if isinstance(other, Value) else Value(other)
        out = Value(self.data * other.data, (self, other), "*")
        def _backward():
            # TODO: each input gets out.grad times the *other* input's data
            raise NotImplementedError("Exercise 3a")
        out._backward = _backward
        return out

    def relu(self):
        out = Value(max(0.0, self.data), (self,), "relu")
        def _backward():
            # TODO: pass out.grad through only if the input was positive
            raise NotImplementedError("Exercise 3b")
        out._backward = _backward
        return out

    def __pow__(self, k):   # k is a plain number
        out = Value(self.data ** k, (self,), f"**{k}")
        def _backward():
            # TODO: d(a^k)/da = k * a^(k-1)
            raise NotImplementedError("Exercise 3c")
        out._backward = _backward
        return out

    # conveniences
    def __neg__(self): return self * -1
    def __sub__(self, other): return self + (-other if isinstance(other, Value) else Value(-other))
    __radd__ = __add__
    __rmul__ = __mul__
    def __repr__(self): return f"Value(data={self.data:.4g}, grad={self.grad:.4g})"

## 4 · The backward pass
To run backprop, call each node's `_backward` **in reverse order**: a node may only push its gradient to its parents once it has received gradient from *all* the nodes that use it. A **topological order** (every node after the nodes it was made from) reversed guarantees that.

The topological sort is written for you. Finish `backward`: seed the output's gradient with $\frac{\partial L}{\partial L} = 1$, then walk the order backwards.

In [ ]:
def topo_order(root):
    order, seen = [], set()
    def visit(n):
        if id(n) not in seen:
            seen.add(id(n))
            for p in n._prev:
                visit(p)
            order.append(n)          # n comes after everything it was made from
    visit(root)
    return order

def backward(root):
    # TODO: set root.grad = 1.0, then call _backward() on every node in reversed(topo_order(root))
    raise NotImplementedError("Exercise 4")

**Check 3 + 4.** The lesson's network, the branch example ($L = 2x + x^2$ at $x = 3$) and a numerical check on a messier expression.

In [ ]:
# the tiny network from the lesson
x, w, v, y = Value(1.5), Value(2.0), Value(0.5), Value(2.5)
L = (v * (w * x).relu() - y) ** 2
backward(L)
assert abs(L.data - 1.0) < 1e-12
assert abs(w.grad - (-1.5)) < 1e-12 and abs(v.grad - (-6.0)) < 1e-12 and abs(x.grad - (-2.0)) < 1e-12

# a branch: x is used twice, so its gradients must add up
x = Value(3.0)
L = 2 * x + x ** 2
backward(L)
assert L.data == 15.0 and x.grad == 8.0, x.grad

# a messier expression, checked numerically
def f(p):
    a, b, c = p
    return (((a * b + c) ** 2) * a - b * c + (a * c).relu()) if isinstance(a, Value) else \
           (((a * b + c) ** 2) * a - b * c + max(0.0, a * c))
vals = [0.7, -1.3, 2.1]
vs = [Value(t) for t in vals]
out = f(vs)
backward(out)
num = numerical_grad(f, vals)
assert np.allclose([t.grad for t in vs], num, atol=1e-5), ([t.grad for t in vs], num)
print("✓ exercises 3 and 4: your engine matches the lesson and the numerical gradients")

## 5 · Train with your engine
Now use it like a real framework. Fit $\hat y = v\cdot\mathrm{relu}(w\,x + b)$ to a few points with gradient descent (P2.2), using **your** `backward`.

One trap: because gradients accumulate with `+=`, you must **reset** every parameter's `.grad` to 0 before each backward pass. (PyTorch has exactly the same rule: `optimizer.zero_grad()`.)

In [ ]:
xs = [0.5, 1.0, 1.5, 2.0, 2.5]
ys = [1.1, 2.0, 2.9, 4.1, 5.0]   # roughly y = 2x

In [ ]:
def train(steps=200, lr=0.02):
    w, b, v = Value(0.5), Value(0.1), Value(0.5)
    params = [w, b, v]
    losses = []
    for _ in range(steps):
        # mean squared error over the 5 points, built from Values
        L = sum(((v * (w * xi + b).relu() - yi) ** 2 for xi, yi in zip(xs, ys)), Value(0.0)) * (1 / len(xs))
        # TODO: 1) reset every p.grad to 0, 2) backward(L), 3) p.data -= lr * p.grad for every p
        raise NotImplementedError("Exercise 5")
        losses.append(L.data)
    return params, losses

**Check 5**

In [ ]:
params, losses = train()
print("loss: start %.3f → end %.4f" % (losses[0], losses[-1]))
assert losses[-1] < 0.05 * losses[0], "the loss should fall by more than 20×"
assert all(a >= b - 1e-12 for a, b in zip(losses, losses[1:])), "with this small learning rate the loss should never go up"
print("✓ exercise 5")

## Experiment
1. In `__add__`, change `self.grad += ...` to `self.grad = ...` (and the same for `other`), re-run the class cell and then `backward` on `2 * x + x ** 2`. What gradient do you get for $x = 3$, and why is it wrong? *(Hint: which path's contribution survives?)* Put the `+=` back afterwards.
2. In `train`, delete the line that resets `p.grad`. What happens to the loss curve?
3. Count the nodes in `topo_order(L)` for the training loss. Each one is visited once forward and once backward. How does that compare with the number of extra forward passes numerical gradients would need?

In [ ]:
# scratch space for the experiments
x = Value(3.0)
L = 2 * x + x ** 2
backward(L)
print("x.grad =", x.grad, "| nodes in the graph:", len(topo_order(L)))

## Recap
- Backprop = the chain rule applied node by node, in **reverse topological order**.
- Every operation only needs its **local derivative**; values from the forward pass are stored for it.
- Gradients from different paths **add up**, so reset them before every backward pass.
- Numerical gradients are slow, but perfect for **checking** an implementation.

Back to the course and hit **Mark complete** to add the review cards.

---
# Solutions
Only peek after a real attempt.

### Exercise 1

In [ ]:
# Solution (copy into the exercise cell above)
def forward_backward(x, w, v, y):
    z = w * x
    h = max(0.0, z)
    yhat = v * h
    L = (yhat - y) ** 2
    dyhat = 2 * (yhat - y)
    dv = dyhat * h
    dh = dyhat * v
    dz = dh * (1.0 if z > 0 else 0.0)
    dw = dz * x
    return L, dw, dv

### Exercise 2

In [ ]:
# Solution (copy into the exercise cell above)
def numerical_grad(f, params, eps=1e-6):
    grads = []
    for i in range(len(params)):
        up = list(params); up[i] += eps
        dn = list(params); dn[i] -= eps
        grads.append((f(up) - f(dn)) / (2 * eps))
    return grads

### Exercise 3

In [ ]:
# Solution (copy into the exercise cell above)
class Value:
    def __init__(self, data, _prev=(), _op=""):
        self.data = float(data)
        self.grad = 0.0
        self._prev = _prev
        self._op = _op
        self._backward = lambda: None

    def __add__(self, other):
        other = other if isinstance(other, Value) else Value(other)
        out = Value(self.data + other.data, (self, other), "+")
        def _backward():
            self.grad += 1.0 * out.grad
            other.grad += 1.0 * out.grad
        out._backward = _backward
        return out

    def __mul__(self, other):
        other = other if isinstance(other, Value) else Value(other)
        out = Value(self.data * other.data, (self, other), "*")
        def _backward():
            self.grad += other.data * out.grad
            other.grad += self.data * out.grad
        out._backward = _backward
        return out

    def relu(self):
        out = Value(max(0.0, self.data), (self,), "relu")
        def _backward():
            self.grad += (1.0 if self.data > 0 else 0.0) * out.grad
        out._backward = _backward
        return out

    def __pow__(self, k):
        out = Value(self.data ** k, (self,), f"**{k}")
        def _backward():
            self.grad += k * self.data ** (k - 1) * out.grad
        out._backward = _backward
        return out

    def __neg__(self): return self * -1
    def __sub__(self, other): return self + (-other if isinstance(other, Value) else Value(-other))
    __radd__ = __add__
    __rmul__ = __mul__
    def __repr__(self): return f"Value(data={self.data:.4g}, grad={self.grad:.4g})"

### Exercise 4

In [ ]:
# Solution (copy into the exercise cell above)
def topo_order(root):
    order, seen = [], set()
    def visit(n):
        if id(n) not in seen:
            seen.add(id(n))
            for p in n._prev:
                visit(p)
            order.append(n)
    visit(root)
    return order

def backward(root):
    root.grad = 1.0
    for node in reversed(topo_order(root)):
        node._backward()

### Exercise 5

In [ ]:
# Solution (copy into the exercise cell above)
def train(steps=200, lr=0.02):
    w, b, v = Value(0.5), Value(0.1), Value(0.5)
    params = [w, b, v]
    losses = []
    for _ in range(steps):
        L = sum(((v * (w * xi + b).relu() - yi) ** 2 for xi, yi in zip(xs, ys)), Value(0.0)) * (1 / len(xs))
        for p in params:
            p.grad = 0.0
        backward(L)
        for p in params:
            p.data -= lr * p.grad
        losses.append(L.data)
    return params, losses